In [292]:
import pandas as pd
import numpy as np
import sklearn as sk
from sklearn import metrics
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt

In [293]:
mida = pd.read_csv("/content/MIDA 5.0.csv") ## Interstate conflict, record per conflict (1816-2014)
midb = pd.read_csv("/content/midb.csv") ## Interstate conflict, record per country involved in conflict (1816-2014)
nmc =  pd.read_csv("/content/NMC-60-abridged.csv") ## Capabilities of war (1816-2016)
vdem = pd.read_csv("/content/V-Dem-CY-Core-v16.csv") ## Democracy indexes (1789-2025)
orgviol = pd.read_csv("/content/organizedviolencecy_v25_1.csv") # Organize intrastate violence (1989-2024)
cow_ccode = pd.read_csv("/content/COW-country-codes.csv") # Correlates of war country codes

In [294]:
# Selecting only important variables and renaming some columns for merging later
mida = mida[["dispnum", "styear", "endyear", "fatality", "fatalpre", "maxdur", "mindur", "hostlev", "recip", "numa", "numb", "ongo2014"]]
nmc = nmc[["stateabb", "ccode", "year", "cinc"]]
vdem = vdem[["country_name", "country_text_id", "year", "v2x_polyarchy", "v2x_libdem", "v2x_partipdem", "v2x_delibdem", "v2x_egaldem"]]
vdem = vdem.rename(columns={"country_text_id" : "stateabb"})
orgviol = orgviol[["country_id_cy", "region_cy", "year_cy", "sb_exist_cy", "sb_dyad_count_cy", "sb_dyad_ids_cy", "sb_dyad_names_cy", "sb_intrastate_exist_cy", "sb_intrastate_dyad_count_cy", "sb_intrastate_dyad_ids_cy", "sb_intrastate_main_govt_inv_incomp_cy", "sb_interstate_exist_cy", "sb_interstate_dyad_count_cy", "sb_interstate_dyad_ids_cy", "sb_interstate_main_govt_inv_incomp_cy", "ns_exist_cy", "ns_dyad_count_cy", "ns_dyad_ids_cy", "os_exist_cy", "os_dyad_count_cy", "os_dyad_ids_cy", "os_main_govt_inv_cy", "os_nsgroup_inv_cy", "cumulative_total_deaths_parties_in_orgvio_cy", "cumulative_total_deaths_civilians_in_orgvio_cy", "cumulative_total_deaths_unknown_in_orgvio_cy", "cumulative_total_deaths_in_orgvio_best_cy"]]
orgviol = orgviol.rename(columns={"country_id_cy" : "ccode", "year_cy" : "year", "cumulative_total_deaths_parties_in_orgvio_cy" : "ctd_parties", "cumulative_total_deaths_civilians_in_orgvio_cy" : "ctd_civ", "cumulative_total_deaths_unknown_in_orgvio_cy" : "ctd_unk", "cumulative_total_deaths_in_orgvio_best_cy" : "ctd_best"})
cow_ccode = cow_ccode.drop_duplicates()

In [295]:
# Removed max and min duration for mida conflicts and combined them into a mean duration variable
mida["meandur"] = mida.apply(lambda row: (row["maxdur"] + row["mindur"])/2, axis=1)
mida = mida.drop(columns=["maxdur", "mindur"])
mida

,dispnum,styear,endyear,fatality,fatalpre,hostlev,recip,numa,numb,ongo2014,meandur
0,2,1902,1903,0,0,3,0,1,1,0,193.0
1,3,1913,1913,0,0,3,0,1,1,0,177.0
2,4,1946,1946,2,-9,4,1,1,1,0,183.0
3,7,1951,1952,2,-9,4,1,1,1,0,106.0
4,8,1856,1857,6,-9,5,1,1,1,0,242.0
...,...,...,...,...,...,...,...,...,...,...,...
2431,4722,2011,2011,0,0,3,1,1,1,0,1.0
2432,4723,2012,2012,0,0,3,1,1,1,0,1.0
2433,4724,2013,2013,0,0,3,0,2,1,0,2.0
2434,4725,2014,2014,0,0,3,0,1,1,1,1.0


In [296]:
# orgviol and nmc have same ccode
df1 = pd.merge(nmc, orgviol, on=["year", "ccode"], how="outer")

In [297]:
# Some stateabb are na even though country code exists
ccode_stateabb = df1[["ccode", "stateabb"]].dropna().drop_duplicates()
ccmap = dict(zip(ccode_stateabb["ccode"], ccode_stateabb["stateabb"]))

def get_stateabb(row):
  ccode = row["ccode"]
  if ccode in ccmap:
    return ccmap[ccode]
  else:
    return pd.NA

df1["stateabb"] = df1.apply(get_stateabb, axis=1)

In [298]:
def get_ccode_cow(row):
  ccode = row['ccode']
  if ccode in cow_ccode['CCode'].values:
    return cow_ccode[cow_ccode["CCode"]==ccode]['StateNme'].tolist()[0]
  else:
    return pd.NA

new_ccode = df1.apply(get_ccode_cow, axis=1)

In [299]:
#figure out why those 108 don't exist
df1[df1['stateabb'].isna()]['ccode'].unique() # These three countires dont have state abreviations, most likely because these countries don't have a military

array([971, 972, 973])

In [300]:
# Combining nmc, vdem and orgviol into one dataframe
df = pd.merge(df1, vdem, on=["year", "stateabb"], how="outer")

stateabbmap = dict(zip(ccode_stateabb["stateabb"], ccode_stateabb["ccode"]))

# Cleaning up some of the missing country codes
def get_ccode(row):
  stateabb = row["stateabb"]
  if stateabb in stateabbmap:
    return stateabbmap[stateabb]
  else:
    return pd.NA

df["ccode"] = df.apply(get_ccode, axis=1)

In [301]:
#Use MIDB to get state sinvolved in MIDA conflicts
# Adds country codes for side a and side b to each conflict record
midb = midb[["dispnum", "ccode","stabb", "sidea"]]

mida["acountries"] = mida.apply(lambda row: midb[((row["dispnum"] == midb["dispnum"]) & (midb["sidea"] == 1))]["ccode"].to_numpy(), axis=1)
mida["bcountries"] = mida.apply(lambda row: midb[((row["dispnum"] == midb["dispnum"]) & (midb["sidea"] == 0))]["ccode"].to_numpy(), axis=1)

In [302]:
#Combine interstate and intrastate sb_main_govt

In [303]:
df.to_csv('fow.csv', index=False) # Only fully overlapping information is from 1989-2016
mida.to_csv('mida.csv', index=False) # Includes involved countries country code and side

In [304]:
conflicts = df[df["year"] >= 1989]
conflicts.columns

Index(['stateabb', 'ccode', 'year', 'cinc', 'region_cy', 'sb_exist_cy',
       'sb_dyad_count_cy', 'sb_dyad_ids_cy', 'sb_dyad_names_cy',
       'sb_intrastate_exist_cy', 'sb_intrastate_dyad_count_cy',
       'sb_intrastate_dyad_ids_cy', 'sb_intrastate_main_govt_inv_incomp_cy',
       'sb_interstate_exist_cy', 'sb_interstate_dyad_count_cy',
       'sb_interstate_dyad_ids_cy', 'sb_interstate_main_govt_inv_incomp_cy',
       'ns_exist_cy', 'ns_dyad_count_cy', 'ns_dyad_ids_cy', 'os_exist_cy',
       'os_dyad_count_cy', 'os_dyad_ids_cy', 'os_main_govt_inv_cy',
       'os_nsgroup_inv_cy', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best',
       'country_name', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem',
       'v2x_delibdem', 'v2x_egaldem'],
      dtype='object')

In [305]:
# Linear Regression - Predict the following for fow
df.columns
#"cumulative_total_deaths_parties_in_orgvio_cy", "cumulative_total_deaths_civilians_in_orgvio_cy", "cumulative_total_deaths_unknown_in_orgvio_cy", "cumulative_total_deaths_in_orgvio_best_cy"

Index(['stateabb', 'ccode', 'year', 'cinc', 'region_cy', 'sb_exist_cy',
       'sb_dyad_count_cy', 'sb_dyad_ids_cy', 'sb_dyad_names_cy',
       'sb_intrastate_exist_cy', 'sb_intrastate_dyad_count_cy',
       'sb_intrastate_dyad_ids_cy', 'sb_intrastate_main_govt_inv_incomp_cy',
       'sb_interstate_exist_cy', 'sb_interstate_dyad_count_cy',
       'sb_interstate_dyad_ids_cy', 'sb_interstate_main_govt_inv_incomp_cy',
       'ns_exist_cy', 'ns_dyad_count_cy', 'ns_dyad_ids_cy', 'os_exist_cy',
       'os_dyad_count_cy', 'os_dyad_ids_cy', 'os_main_govt_inv_cy',
       'os_nsgroup_inv_cy', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best',
       'country_name', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem',
       'v2x_delibdem', 'v2x_egaldem'],
      dtype='object')

# Use democratic indeces to predict severity of intrastate conflict

In [306]:
step1 = conflicts[['v2x_polyarchy','v2x_libdem','v2x_partipdem','v2x_delibdem','v2x_egaldem', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
step1 = step1.dropna()

X = step1[['v2x_polyarchy', 'v2x_libdem','v2x_partipdem','v2x_delibdem','v2x_egaldem']]
y = step1[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True)

In [318]:
# Trained model with cross validation to see how reliable it is
linear_models_metrics = pd.DataFrame(columns=["pred_var", "rmse", "normalized_rmse", "r^2"])

for pred_var in y_train.columns:
  linear_model = sk.linear_model.LinearRegression()
  scores = cross_validate(linear_model, X_train, y_train[pred_var], scoring={"nmse": "neg_mean_squared_error",
    "r2": "r2"}, cv=5)

  rmse_score = np.sqrt(-scores['test_nmse']).mean()
  linear_r2_score = scores['test_r2'].mean()
  normalized_rmse = rmse_score / y_train[pred_var].std()

  linear_models_metrics.loc[len(linear_models_metrics)] = [pred_var, rmse_score.item(), normalized_rmse.item(), linear_r2_score.item()]

linear_models_metrics

,pred_var,rmse,normalized_rmse,r^2
0,ctd_parties,3734.892605,0.873487,-0.081653
1,ctd_civ,8515.489798,0.513839,-1.376324
2,ctd_unk,1315.434184,0.882537,0.003548
3,ctd_best,11384.566298,0.655532,-0.099424


In [319]:
# Tested final model on test data
final_models = {}

for pred_var in y_train.columns:
    model = sk.linear_model.LinearRegression()
    model.fit(X_train, y_train[pred_var])
    final_models[pred_var] = model

test_metrics = pd.DataFrame(columns=["pred_var", "rmse", "normalized_rmse", "r^2"])

for pred_var in y_test.columns:
    model = final_models[pred_var]

    y_pred = model.predict(X_test)

    rmse = np.sqrt(sk.metrics.mean_squared_error(y_test[pred_var], y_pred))
    r2 = sk.metrics.r2_score(y_test[pred_var], y_pred)
    normalized_rmse = rmse / y_test[pred_var].std()

    test_metrics.loc[len(test_metrics)] = [
        pred_var,
        rmse,
        normalized_rmse,
        r2
    ]

test_metrics

,pred_var,rmse,normalized_rmse,r^2
0,ctd_parties,7575.266059,0.994626,0.008895
1,ctd_civ,1625.417026,1.104757,-0.222739
2,ctd_unk,1158.607869,0.993101,0.011930
3,ctd_best,8913.754669,0.997059,0.004038


# Use other factors of war and region to predict severity of intrastate conflict

In [320]:
step2 = conflicts[['ccode','year','cinc','region_cy', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
step2 = step2.dropna()

X = step2[['ccode','year','cinc','region_cy']]
y = step2[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), ["region_cy"])
    ],
    remainder="passthrough"
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.fit_transform(X_test)

In [321]:
# Trained model with cross validation to see how reliable it is
linear_models_metrics = pd.DataFrame(columns=["pred_var", "rmse", "normalized_rmse", "r^2"])

for pred_var in y_train.columns:
  linear_model = sk.linear_model.LinearRegression()
  scores = cross_validate(linear_model, X_train_enc, y_train[pred_var], scoring={"nmse": "neg_mean_squared_error",
    "r2": "r2"}, cv=5)

  rmse_score = np.sqrt(-scores['test_nmse']).mean()
  linear_r2_score = scores['test_r2'].mean()
  normalized_rmse = rmse_score / y_train[pred_var].std()

  linear_models_metrics.loc[len(linear_models_metrics)] = [pred_var, rmse_score.item(), normalized_rmse.item(), linear_r2_score.item()]

linear_models_metrics

,pred_var,rmse,normalized_rmse,r^2
0,ctd_parties,1561.947699,0.950452,0.005514
1,ctd_civ,6241.801804,0.519245,-0.242799
2,ctd_unk,858.939106,0.852997,-0.003544
3,ctd_best,7564.367208,0.615060,-0.021887


In [322]:
# Tested final model on test data
final_models = {}

for pred_var in y_train.columns:
    model = sk.linear_model.LinearRegression()
    X_train_enc = preprocessor.fit_transform(X_train)

    model.fit(X_train_enc, y_train[pred_var])
    final_models[pred_var] = model

test_metrics = pd.DataFrame(columns=["pred_var", "rmse", "normalized_rmse", "r^2"])

for pred_var in y_test.columns:
    model = final_models[pred_var]

    y_pred = model.predict(X_test_enc)

    rmse = np.sqrt(sk.metrics.mean_squared_error(y_test[pred_var], y_pred))
    r2 = sk.metrics.r2_score(y_test[pred_var], y_pred)
    normalized_rmse = rmse / y_test[pred_var].std()

    test_metrics.loc[len(test_metrics)] = [
        pred_var,
        rmse,
        normalized_rmse,
        r2
    ]

test_metrics

,pred_var,rmse,normalized_rmse,r^2
0,ctd_parties,2030.991589,0.996845,0.005340
1,ctd_civ,832.610066,1.167397,-0.364133
2,ctd_unk,330.809138,0.997943,0.003147
3,ctd_best,2584.878753,1.001433,-0.003837


The results of both linear models are very unstable changing a lot with different runs of the data. Their coefficient of determination is also not very high except for the ctd_div which can vary from 0.15 to 0.5. From the cross validation set we realized that the model is too simple to deal with the data and has even given us (in the first models training set) a r^2 of -1.9

In [323]:
# Mida dataset predict fatality and hostlevel using other variables (Linear Model)

In [324]:
# Tree based aproach, knn, nn
step1 = conflicts[['v2x_polyarchy','v2x_libdem','v2x_partipdem','v2x_delibdem','v2x_egaldem', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
step1 = step1.dropna()

X = step1[['v2x_polyarchy', 'v2x_libdem','v2x_partipdem','v2x_delibdem','v2x_egaldem']]
y = step1[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True)

In [325]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train['ctd_civ'])
y_pred = rf.predict(X_test)

In [326]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb.fit(X_train, y_train['ctd_civ'])
y_pred = gb.predict(X_test)

In [329]:
def evaluate(model, X_test, y_test):
    pred = pd.DataFrame(model.predict(X_test))
    rmse = np.sqrt(metrics.mean_squared_error(y_test, pred))
    r2 = sk.metrics.r2_score(y_test, pred)
    return rmse, r2

print("RF:", evaluate(rf, X_test, y_test['ctd_civ']))
print("GB:", evaluate(gb, X_test, y_test['ctd_civ']))

RF: (np.float64(8945.478906488554), -499.86466238889335)
GB: (np.float64(17444.520480062813), -1903.7205692991758)


ns_dyad_count has a significant relationship with death count, and unknown count in particular – so probably that means it's more chaotic and harder to collect data

In [331]:
df[['sb_exist_cy',
       'sb_dyad_count_cy',
       'sb_intrastate_exist_cy', 'sb_intrastate_dyad_count_cy',
       'sb_intrastate_main_govt_inv_incomp_cy',
       'sb_interstate_exist_cy', 'sb_interstate_dyad_count_cy',
        'sb_interstate_main_govt_inv_incomp_cy',
       'ns_exist_cy', 'ns_dyad_count_cy',  'os_exist_cy',
       'os_dyad_count_cy', 'os_main_govt_inv_cy',
       'os_nsgroup_inv_cy', 'ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']].corr()[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]

,ctd_parties,ctd_civ,ctd_unk,ctd_best
sb_exist_cy,0.154245,0.042599,0.128168,0.100690
sb_dyad_count_cy,0.190020,0.029636,0.146087,0.101891
sb_intrastate_exist_cy,0.140725,0.042915,0.129261,0.096737
sb_intrastate_dyad_count_cy,0.184001,0.030071,0.136176,0.099443
sb_intrastate_main_govt_inv_incomp_cy,0.150115,0.045621,0.138577,0.103109
sb_interstate_exist_cy,0.134414,0.007689,0.155518,0.064649
sb_interstate_dyad_count_cy,0.132471,0.007540,0.152307,0.063591
sb_interstate_main_govt_inv_incomp_cy,0.121783,0.005453,0.160965,0.059029
ns_exist_cy,0.142747,0.015246,0.206028,0.078957
ns_dyad_count_cy,0.197698,0.025082,0.330283,0.117157


In [336]:
from sklearn.neighbors import KNeighborsClassifier
step3 = conflicts[['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']]

dem_ind_avg = (conflicts['v2x_polyarchy'] + conflicts['v2x_libdem'] + conflicts['v2x_partipdem'] + conflicts['v2x_delibdem'] + conflicts['v2x_egaldem'])/5
step3['dem_ind_avg'] = dem_ind_avg
step3 = step3.dropna()


for pred_var in ['ctd_parties', 'ctd_civ', 'ctd_unk', 'ctd_best']:
  print(f"Predicting for {pred_var}")
  X = step3[['dem_ind_avg']]
  y = step3[pred_var]
  X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True)

  for neighbors in range(2,8):
    model = KNeighborsClassifier(n_neighbors=neighbors).fit(X_train, y_train)
    print(f"Neighbors {neighbors}")
    print(evaluate(model, X_test, y_test))
  print()

/tmp/ipykernel_2256/3295012227.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  step3['dem_ind_avg'] = dem_ind_avg


Predicting for ctd_parties
Neighbors 2
(np.float64(3169.8046174923), -0.006792746573250907)
Neighbors 3
(np.float64(3168.4553841230963), -0.00593584248545409)
Neighbors 4
(np.float64(3171.084007220109), -0.007605629636762146)
Neighbors 5
(np.float64(3171.7220026740642), -0.008011113993160679)
Neighbors 6
(np.float64(3172.1399565964502), -0.008276792988426873)
Neighbors 7
(np.float64(3172.069390658542), -0.008231934181970324)

Predicting for ctd_civ
Neighbors 2
(np.float64(1578.767855927451), 0.009488798739165638)
Neighbors 3
(np.float64(1580.0674927888226), 0.007857355944537403)
Neighbors 4
(np.float64(1580.048821424425), 0.007880803737083997)
Neighbors 5
(np.float64(1585.4437672861561), 0.0010942196861031261)
Neighbors 6
(np.float64(1585.403127127749), 0.0011454295362189892)
Neighbors 7
(np.float64(1590.8286940150588), -0.005702829110519669)

Predicting for ctd_unk
Neighbors 2
(np.float64(792.689328808813), -0.002971713163061107)
Neighbors 3
(np.float64(796.2458611854006), -0.01199190